# 01 — I segnali dall'HTML di ogni articolo (tentativo 09, cambio di paradigma)

Per ogni articolo del dataset finale (`../Dataset/versioni/dataset_finale.csv`: anche le testate di solo training)
si legge la sua pagina HTML grezza, salvata dallo scraping in `../Dataset/NuovoScraping/html/<dominio>.jsonl.gz`, e
si estraggono due famiglie di segnali, osservabili per qualunque sito. Nessun metadato NewsGuard entra qui.

**Link citati.** I link che trafilatura conserva nel **testo dell'articolo** (lo stesso estrattore usato per il
testo, con i link attivi): sono le fonti che l'articolo cita, non i menu del sito. Per ogni link il dominio
registrato (`news.bbc.co.uk` → `bbc.co.uk`); si esclude il sito stesso. I link verso siti dello stesso gruppo si
escludono più avanti, in classificazione, perché i gruppi dipendono dal fold.

**Pagina.** Com'è fatta la pagina che contiene l'articolo:

- servizi di **terze parti** (script, iframe e immagini caricati da altri domini), per categoria secondo la lista
  pubblica di Disconnect (pubblicità, analytics, social, fingerprinting, contenuti); le **reti pubblicitarie**
  riconoscibili nel codice (AdSense, Taboola, Outbrain, Revcontent, MGID…);
- **trasparenza**: link verso pagine "chi siamo", contatti, rettifiche, etica e standard, redazione, proprietà e
  finanziamento, pubblicità, donazioni, newsletter;
- **articolo**: firma dell'autore, data di pubblicazione, dati strutturati `NewsArticle` (e `OpinionNewsArticle`),
  paywall dichiarato, sezione commenti, banner dei cookie;
- **contenuti incorporati**: video (YouTube/Vimeo; Rumble/BitChute/Odysee/Brighteon) e social;
- dimensioni: KB della pagina, script, immagini, link.

Tutte regole fisse nel codice. Uscite (fuori da git): `../Dataset/segnali/pagine.csv.gz` (una riga per articolo) e
`../Dataset/segnali/link_citati.csv.gz` (articolo, dominio citato, quante volte); riepilogo in
`../Results/2-segnali/01_riepilogo.csv`.

In [ ]:
import gzip
import json
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import trafilatura
from joblib import Parallel, delayed
from lxml import html as lh

from comune import HTML, INCORPORATI, RESULTS, SEGNALI, TRASPARENZA, VERSIONI, categorie_disconnect, dominio

In [ ]:
N_JOBS = 6
PROVA = False             # True: 8 testate, uscite in ../Dataset/segnali/prova

In [ ]:
OUT = SEGNALI / "prova" if PROVA else SEGNALI
OUT.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(parents=True, exist_ok=True)
art = pd.read_csv(VERSIONI / "dataset_finale.csv", usecols=["article_id", "domain", "url"])
if PROVA:
    art = art[art.domain.isin(sorted(art.domain.unique())[:8])]
CAT = categorie_disconnect()
RETI_PUBBLICITARIE = r"adsbygoogle|googlesyndication|doubleclick|taboola|outbrain|revcontent|mgid|zergnet|content\.ad|amazon-adsystem|criteo|adthrive|mediavine|ezoic|propellerads|media\.net"
RX_TRASP = {k: re.compile(v, re.I) for k, v in TRASPARENZA.items()}
RX_INC = {k: re.compile(v, re.I) for k, v in INCORPORATI.items()}
print(f"{len(art)} articoli di {art.domain.nunique()} testate; {len(CAT)} domini nella lista di Disconnect")

## L'estrazione, testata per testata (in parallelo)

In [ ]:
def segnali_pagina(h, proprio):
    r = {"KB pagina": len(h) / 1024}
    try:
        doc = lh.fromstring(h)
    except Exception:
        return r
    # servizi di terze parti caricati dalla pagina
    terzi = set()
    for e in doc.xpath("//script[@src]|//iframe[@src]|//img[@src]|//link[@href and @rel='preconnect']"):
        d = dominio(e.get("src") or e.get("href") or "")
        if d and d != proprio:
            terzi.add(d)
    cat = Counter(CAT.get(d, "altro") for d in terzi)
    r.update({"terze parti": len(terzi), "terze parti: pubblicita": cat["Advertising"], "terze parti: analytics": cat["Analytics"],
              "terze parti: social": cat["Social"], "terze parti: fingerprinting": cat["FingerprintingInvasive"] + cat["FingerprintingGeneral"],
              "terze parti: contenuti": cat["Content"]})
    r["reti pubblicitarie nel codice"] = len(set(m.lower() for m in re.findall(RETI_PUBBLICITARIE, h, re.I)))
    # trasparenza: link della pagina verso pagine del sito
    link = [(a.get("href") or "", (a.text_content() or "").strip()[:60]) for a in doc.xpath("//a[@href]")]
    r["link nella pagina"] = len(link)
    for k, rx in RX_TRASP.items():
        r[f"pagina: {k}"] = int(any(rx.search(href) or rx.search(testo) for href, testo in link))
    # l'articolo
    r["firma dell'autore"] = int(bool(doc.xpath("//meta[@name='author' and normalize-space(@content)!='']|//a[@rel='author']|//*[@itemprop='author']"))
                                 or bool(re.search(r'"author"\s*:', h)) or bool(doc.xpath("//*[contains(@class,'byline')]")))
    r["data di pubblicazione"] = int(bool(doc.xpath("//meta[@property='article:published_time']|//time[@datetime]")) or bool(re.search(r'"datePublished"', h)))
    r["dati strutturati NewsArticle"] = int(bool(re.search(r'"@type"\s*:\s*"(News|Reportage|Analysis|Report)Article"', h)))
    r["dichiarato come opinione"] = int(bool(re.search(r'"@type"\s*:\s*"OpinionNewsArticle"|class="[^"]*\b(opinion|op-ed|commentary)\b', h, re.I)))
    r["paywall dichiarato"] = int(bool(re.search(r'"isAccessibleForFree"\s*:\s*"?false', h, re.I)))
    r["sezione commenti"] = int(bool(re.search(r"disqus|id=\"comments\"|class=\"[^\"]*comments", h, re.I)))
    r["banner dei cookie"] = int(bool(re.search(r"onetrust|cookiebot|cookieconsent|didomi|usercentrics|quantcast\.mgr|consentmanager", h, re.I)))
    # contenuti incorporati (iframe e link a piattaforme)
    srcs = " ".join(e.get("src") or "" for e in doc.xpath("//iframe[@src]"))
    for k, rx in RX_INC.items():
        r[f"incorporati: {k}"] = len(rx.findall(srcs))
    r["script"] = len(doc.xpath("//script")); r["immagini"] = len(doc.xpath("//img"))
    return r


def una_testata(dom, righe):
    p = HTML / f"{dom}.jsonl.gz"
    pagine = {}
    if p.exists():
        with gzip.open(p, "rt", encoding="utf-8") as f:
            for l in f:
                if l.strip():
                    x = json.loads(l); pagine[x["url"]] = x.get("html") or ""
    proprio = dominio("http://" + dom)
    righe_p, righe_l = [], []
    for aid, url in zip(righe.article_id, righe.url):
        h = pagine.get(url, "")
        if not h:
            righe_p.append({"article_id": aid, "domain": dom, "html": 0}); continue
        md = trafilatura.extract(h, include_links=True, output_format="markdown") or ""
        citati = [d for d in (dominio(u) for u in re.findall(r"\]\((https?://[^)\s]+)\)", md)) if d and d != proprio]
        righe_p.append({"article_id": aid, "domain": dom, "html": 1, "link citati nel testo": len(citati),
                        "parole del testo estratto": len(md.split()), **segnali_pagina(h, proprio)})
        righe_l += [{"article_id": aid, "domain": dom, "citato": d, "volte": n} for d, n in Counter(citati).items()]
    return righe_p, righe_l


t0 = time.time()
gruppi = list(art.groupby("domain"))
ris = Parallel(n_jobs=N_JOBS, batch_size=4)(delayed(una_testata)(d, g) for d, g in gruppi)
pagine = pd.DataFrame([r for rp, _ in ris for r in rp]); link = pd.DataFrame([r for _, rl in ris for r in rl])
pagine.to_csv(OUT / "pagine.csv.gz", index=False); link.to_csv(OUT / "link_citati.csv.gz", index=False)
print(f"{len(pagine)} articoli in {(time.time() - t0) / 60:.1f} min; con HTML {pagine.html.mean():.1%}; "
      f"{len(link)} coppie articolo-dominio citato, {link.citato.nunique() if len(link) else 0} domini citati diversi")

## Riepilogo

In [ ]:
num = pagine.drop(columns=["article_id", "domain", "html"]).select_dtypes("number")
rie = pd.DataFrame({"media": num.mean(), "mediana": num.median(), "articoli con valore > 0": (num > 0).mean()}).round(3)
if not PROVA:
    rie.to_csv(RESULTS / "01_riepilogo.csv")
display(rie)
if len(link):
    print("domini più citati (articoli):", link.groupby("citato").article_id.nunique().sort_values(ascending=False).head(25).to_dict())